# Databases - week 12
working with the mysql package

In [ ]:
! python -m pip install mysql-connector-python

## Connect to the database

In [1]:
import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()

#ladida this is where we do stuff!

cursor.close()
dbconnection.close()

## create operations

In [2]:
import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()
query = "INSERT INTO actors(name) VALUES('Kevin Hart');"
cursor.execute(query)
dbconnection.commit() #as if you actually press start after entering the query!
#import to COMMIT!!

cursor.close()
dbconnection.close()

In [3]:
import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()
query = "INSERT INTO actors(name) VALUES(%s);"

data = [["Leonardo Dicaprio"], ["Ryan Gosling"], ["Angelina Jolie"]]

cursor.executemany(query, data)
dbconnection.commit()

cursor.close()
dbconnection.close()

In [4]:
import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()
query = "INSERT INTO movies(title, synopsis) VALUES(%s, %s);"

data = ("Catch me if you can", "Homeless guy trying to escape")
cursor.execute(query, data)

movies = [("The fall guy", "A stunt actor tries not to fall in love"), ("Maleficent", "Bad witch, trying to bring death")]
for movie in movies:
    cursor.execute(query, movie)

dbconnection.commit()
cursor.close()
dbconnection.close()

## READING OPERATIONS

In [5]:
import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()
query = "SELECT * FROM actors;"
cursor.execute(query)

data = cursor.fetchall()
print(data)
for actor in data:
    print(actor) #entire record

for actor in data:
    print(actor[1])

for id, name in data:
    print(id, "of actor:", name)

cursor.close()
dbconnection.close()

[(1, 'Michelle Yeoh'), (2, 'Stephan James'), (3, 'Jamie Lee Curtis'), (4, 'Tom Cruise'), (5, 'Kevin Hart'), (6, 'Leonardo Dicaprio'), (7, 'Ryan Gosling'), (8, 'Angelina Jolie')]
(1, 'Michelle Yeoh')
(2, 'Stephan James')
(3, 'Jamie Lee Curtis')
(4, 'Tom Cruise')
(5, 'Kevin Hart')
(6, 'Leonardo Dicaprio')
(7, 'Ryan Gosling')
(8, 'Angelina Jolie')
Michelle Yeoh
Stephan James
Jamie Lee Curtis
Tom Cruise
Kevin Hart
Leonardo Dicaprio
Ryan Gosling
Angelina Jolie
1 of actor: Michelle Yeoh
2 of actor: Stephan James
3 of actor: Jamie Lee Curtis
4 of actor: Tom Cruise
5 of actor: Kevin Hart
6 of actor: Leonardo Dicaprio
7 of actor: Ryan Gosling
8 of actor: Angelina Jolie


In [6]:
import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()
query = """SELECT movies.title, actors.name from movies 
inner join movie_actor on movies.id = movie_actor.movie_id
inner join actors on actors.id = movie_actor.actor_id;"""

cursor.execute(query)
data = cursor.fetchall()
for movie, actor in data:
    print(actor, "played in", movie)

dbconnection.commit()
cursor.close()
dbconnection.close()

Michelle Yeoh played in Everything Everywhere All at Once
Stephan James played in Everything Everywhere All at Once
Jamie Lee Curtis played in Everything Everywhere All at Once
Tom Cruise played in Top Gun: Maverick


### If you need extra information
Check slides

## UPDATE OPERATIONS

In [9]:
import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()
#cursor.execute("INSERT INTO movies(title) VALUES ('La la land');")
cursor.execute("""UPDATE movies SET synopsis = 'a romantic musical-drama following an aspiring actress 
               (Emma Stone) and a jazz pianist (Ryan Gosling) who fall in love while chasing their dreams 
               in Los Angeles.' 
               WHERE title like 'La la land';""")

dbconnection.commit()
cursor.close()
dbconnection.close()


### SCRAPING DATA FROM SOMEWHERE INTO THE DATABASE
https://en.wikipedia.org/wiki/List_of_Academy_Award%E2%80%93winning_films

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_Academy_Award-winning_films"
headers = {'User-Agent' : 'Anthony'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text)



import mysql.connector

dbconnection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "",
    port = "3308",
    database = "movies_db"
)

cursor = dbconnection.cursor()



#now official scraping starts here:
tables = soup.find_all("table")
rows = tables[0].find_all("tr")
for row in rows:
    td = row.find_all("td")
    if len(td) > 0:
        title = td[0].get_text().strip().replace("'", "")
        year = td[1].get_text().strip()
        awards = td[2].get_text().strip()
        nom = td[3].get_text().strip()
    
        #checking if this stuff is already in the DB?
        query = f"SELECT * from movies where title like '{title}';"
        cursor.execute(query)
        data = cursor.fetchall()
        if len(data) == 0:
            query = f"INSERT into movies(title) values('{title}')"
            cursor.execute(query)
            dbconnection.commit()

            movie_id = cursor.lastrowid
            query = "INSERT into awards(awardshow_id, year, movie_id, awards, nominations) VALUES (%s, %s, %s, %s, %s)"
            values = (1, year, movie_id, awards, nom)
            cursor.execute(query, values)
            dbconnection.commit()

cursor.close()
dbconnection.close()